In [ ]:
import os
from collections import Counter
import numpy as np
import ast
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import PatternFill

### Prepare for tie break

In [ ]:
def annotated_to_majority(part = "part3A"):
    df_ann = pd.read_csv(os.path.join("annotated_data", f"raw/annotations_{part}.csv"))
    df_ann_filtered = df_ann[df_ann.round_id>=0]
    question_cols = [c for c in df_ann.columns if "question" in c]
    responses = {}
    examples_ids = sorted(df_ann_filtered.example_id.unique(), key=lambda x: int(x))
    for example_id in examples_ids:
        df_ann_r = df_ann_filtered[df_ann_filtered.example_id == example_id]
        responses[example_id] = {}
        for col in question_cols:
            response = df_ann_r[col].values
            c_reponsed = Counter(response)
            value, count = c_reponsed.most_common()[0]
            if count == 1:
                value = response
            responses[example_id][col] = (value, count)
    df_tie_break = pd.DataFrame(responses).T
    df_tie_break["example_id"] = np.arange(len(df_tie_break))
    df = pd.read_csv("data/posts_abuse_shuffled.csv")
    df_part = df[df.part == part].reset_index()
    df_part["example_id"] = np.arange(len(df_part))
    df_merged = df_tie_break.merge(df_part, left_on="example_id", right_on="example_id")
    df_merged["example_id"] = np.arange(len(df_merged))
    df_merged.to_csv(f"annotated_data/by_majority/raw/annotations_{part}_majority.csv", index=False)

In [ ]:
annotated_to_majority(part="part3B")

In [ ]:
def color_workbook(part = "part3A"):
    df_merged = pd.read_csv(f"annotated_data/by_majority/raw/annotations_{part}_majority.csv")
    # Create the Workbook and Sheet
    wb = Workbook()
    ws = wb.active
    ws.append(list(df_merged.columns))
    # Convert DataFrame to a list of lists
    data = df_merged.values.tolist()

    # Set cell coloring rules
    yellow_fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')
    green_fill = PatternFill(start_color='00FF00', end_color='00FF00', fill_type='solid')
    orange_fill = PatternFill(start_color='FFA500', end_color='FFA500', fill_type='solid')
    # Define a custom converter for numpy arrays

    # Set the custom converter for numpy arrays
    # Iterate over the data and write to Excel
    for row_idx_, row_data in enumerate(data, start=1):
        ws.append(row_data)  # Write the row to Excel

        # Apply cell coloring rules
        row_idx = row_idx_ + 1 ## skip header row
        for col_idx, cell in enumerate(ws[row_idx], start=1):
            if col_idx > 18:
                continue
            else:
                try:
                    tuple_value = ast.literal_eval(cell.value)
                    if not isinstance(tuple_value, tuple):
                        tuple_value = (tuple_value,)
                except (ValueError, SyntaxError):
                    # Apply yellow fill to the whole row
                    for cell_to_fill in ws[row_idx]:
                        if cell_to_fill.fill != green_fill:
                            cell_to_fill.fill = yellow_fill

                    # Apply green fill to the specific cell
                    cell.fill = green_fill

            if col_idx < 2 and 'irrelevant' in cell.value:
                # Apply orange fill to the whole row
                for cell_to_fill in ws[row_idx]:
                    cell_to_fill.fill = orange_fill

    # Save the Workbook to an Excel file
    wb.save(f'annotated_data/by_majority/colored/annotations_{part}_majority.xlsx')

In [ ]:
color_workbook(part="part3B")

In [ ]:
from typing import Optional


def annotated_to_user(parts: Optional[list] = None):
    if parts is None:
        parts = ["evaluation", "exploration", "part3A", "part3B"]

    df = pd.read_csv("data/posts_abuse_shuffled.csv")
    for part in parts:
        df_part = df[df.part == part].reset_index()
        df_part["round_id"] = np.arange(len(df_part))
        df_ann = pd.read_csv(os.path.join("annotated_data", f"annotations_{part}.csv"))
        for user in df_ann.name.unique():
            df_ann_user_sorted = df_ann[df_ann.name == user].sort_values("round_id")
            df_merged = df_ann_user_sorted.merge(df_part, left_on="round_id", right_on="round_id")
            df_merged.to_csv(f"annotated_data/by_user/annotations_{part}_{user}.csv", index=False)

#### Prepare final dataset

In [ ]:
def tiebreak_to_final(part = "part3A"):
    df = pd.read_csv(f"annotated_data/by_tiebreak/annotations_{part}.csv")
    for col in df.columns:
        if "question" in col and type(df[col].values[0]) == str:
            df[col] = df[col].apply(lambda x: x.replace("(", "").replace(")", "").split(",")[0])

    df.to_csv(f"annotated_data/final/annotations_{part}.csv")
    return df

In [ ]:
tiebreak_to_final("part3B")

,question_0_This post is:,question_1_Total relationship duration,question_2_Current relationship status,question_3_How would you rate the risk level of the woman's relationship based on the information provided in the post?,question_4_Does she mention avoiding social events or outings with friends or family because of him or maybe making excuses for not going out or it's implied from the text?,"question_5_Does she frequently feel anxious in her interactions with him, as if she needs to be careful with her words and actions?",question_6_Does he frequently make her feel guilty?,question_7_Does he frequently express dissatisfaction with her?,question_8_Does he try to manipulate her?,question_9_Does she feel she doesn't have option to leave him?,...,ups,upvote_ratio,downs,URL,over18,TOXICITY,THREAT,PROFANITY,INSULT,IDENTITY_ATTACK
0,'relevant','3+ years long','partners/married','low','no','no','no','no','no','no',...,1,1.00,0,/r/relationship_advice/comments/1328vwu/can_i_...,False,0.072128,0.011068,0.079348,0.018130,0.015958
1,'relevant','3+ years long','boyfriend/girlfriend dating','low','no','no','no','no','no','no',...,1,0.67,0,/r/relationship_advice/comments/1325n7a/im_22f...,False,0.034277,0.007690,0.029460,0.015265,0.006253
2,'relevant','3+ years long','partners/married','low','no','no','no','no','no','no',...,1,1.00,0,/r/relationship_advice/comments/132kkrt/how_do...,False,0.038284,0.006550,0.019007,0.018194,0.005476
3,'relevant','unclear from the post','boyfriend/girlfriend dating','low','no','no','no','no','no','no',...,0,0.25,0,/r/relationship_advice/comments/131infw/i_28f_...,False,0.042162,0.008609,0.036902,0.017228,0.008953
4,'relevant','unclear from the post','boyfriend/girlfriend dating','medium','no','no','no','yes','no','no',...,1,1.00,0,/r/relationship_advice/comments/132eyh0/my_20f...,False,0.111920,0.014393,0.098068,0.031367,0.011395
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,'relevant','3+ years long','partners/married','low','no','no','yes','no','yes','no',...,5,0.86,0,/r/relationship_advice/comments/131glce/i_feel...,False,0.247637,0.012130,0.225927,0.063549,0.052490
116,'relevant','unclear from the post','boyfriend/girlfriend dating','high','no','no','no','no','yes','yes',...,3,0.60,0,/r/relationship_advice/comments/132fczg/22m_bo...,False,0.227128,0.041674,0.181105,0.067046,0.033527
117,'relevant','3+ years long','separated/divorced','high','no','no','no','no','yes','yes',...,6,0.88,0,/r/relationship_advice/comments/132i2zk/i_31f_...,False,0.110990,0.010162,0.080965,0.033130,0.016260
118,'relevant','unclear from the post','boyfriend/girlfriend dating','low','no','no','no','no','no','no',...,2,1.00,0,/r/relationship_advice/comments/1316vnq/my_31f...,False,0.153245,0.009075,0.096806,0.050863,0.019274


##### Concat all files

In [ ]:
root = "annotated_data/final"
dfs = []
for file in os.listdir(root):
    df = pd.read_csv(os.path.join(root, file))
    dfs.append(df)

In [ ]:
cols_all = [list(df.columns) for df in dfs]
final_cols = cols_all[-1]
final_cols = [x for x in final_cols if x != "Unnamed: 0"]

for df in dfs:
    try:
        df = df[final_cols]
    except KeyError as e:
        print(df["part"][0])
        print(e)

evaluation
"['example_id'] not in index"
exploration
"['example_id'] not in index"
part3A
"['example_id'] not in index"
part3B
"['question_4_Does she mention avoiding social events or outings with friends or family because of him or maybe making excuses for not going out?', 'round_id'] not in index"
pre_exploration_t2
"['example_id'] not in index"


In [ ]:
df_all = pd.concat(dfs, axis=0)
df_all.to_csv(os.path.join(root, "final_dataset.csv"), index=False)

In [ ]:
len(df_all)

2579

### prepare prompt dataset

In [40]:
import pandas as pd


with open("prompt.txt", "r") as f:
    text = f.readlines()

ann_df = pd.read_csv("final_dataset_clean.csv")
prompt_df = ann_df.iloc[:, [0, 4, 26,  27]]

In [41]:
prompt_text = '\n'.join(text)

In [42]:
prompt_df = prompt_df.rename(columns={"Unnamed: 0": "example_id", "question_3_How would you rate the risk level of the woman\'s relationship based on the information provided in the post?": "label", "text": "raw_text"})

In [43]:
prompt_df

,example_id,label,title,raw_text
0,0,medium,I (18F) am not sure how to approach my bf (19m...,**sorry I had to repost because the post didn'...
1,1,low,Should I (20f) tell my boyfriend (20m) that I ...,"For context, I come from a pretty traditional ..."
2,3,low,My (F20) bf (M23) liked pics of a girl he foll...,"2 weeks ago she posted 4 pics in a row, each o..."
3,4,low,I (34F) found a video on my husbands (34M) pho...,A few weeks ago my daughter wanted to play som...
4,5,low,I'm (24F) interested in someone (25M) ((or clo...,I don't have any prior relationship experienc...
...,...,...,...,...
271,289,low,I (22F) am doubting my perfect relationship (5...,"Hey guys, so a TW on this but i would love som..."
272,290,low,I (f26) need some help figuring out what to do...,Seriously it�s actually kind of embarrassing a...
273,291,high,I've (24F) tried to leave my husband (25M) 3 t...,My ex(?) and I have had a very difficult relat...
274,293,low,I (19F) don�t know if I should wait for my boy...,English is not my first language.\nFor context...


In [44]:
prompt_df["text"] = prompt_df["title"] + " " + prompt_df["raw_text"]

In [45]:
prompt_df["text"].head(1)

0    I (18F) am not sure how to approach my bf (19m...
Name: text, dtype: object

In [46]:
prompt_df["prompted"] = prompt_text

In [47]:
def replace_placeholders(row):
    title = row['title']
    raw_text = row['raw_text']
    prompted_text = row['prompted']

    splited_title = title.split()
    splited_title_len = len(splited_title)
    text_length = 500-splited_title_len
    splited_text = raw_text.split()
    limited_words = splited_text[:text_length]
    limited_text = ' '.join(limited_words)
    prompted_text = prompted_text.replace('[title]', title).replace("â€˜", "'").replace("â€™", r"'").replace("�", "'")
    prompted_text = prompted_text.replace('[post]', limited_text).replace("â€˜", "'").replace("â€™", r"'").replace("�", "'")
    prompted_text  = prompted_text.replace("â€˜", "'").replace("â€™", r"'").replace("�", "'")
    # return limited_text
    return prompted_text

In [48]:
prompt_df['prompted'] = prompt_df.apply(replace_placeholders, axis=1)

In [49]:
prompt_df = prompt_df.drop(columns=["title", "raw_text"])

In [50]:
## check length
for i, prompt in enumerate(prompt_df['prompted'].values):
    splited_text = prompt.split()
    len_of_text = len(splited_text)
    if len_of_text > 800:
      print(i)

In [51]:
prompt_df.to_csv("prompts_team7.csv")